<a href="https://colab.research.google.com/github/krishhh1089/load-aware-colab/blob/main/Load_Aware_SAGE_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Load-Aware-SAGE"

directories = [
    "artifacts/dataset",
    "artifacts/indexes/bm25",
    "artifacts/indexes/faiss",
    "artifacts/models/random_forest",
    "artifacts/metadata",
    "experiments/oracle_labels",
    "experiments/logs"
]

for directory in directories:
    os.makedirs(
        os.path.join(PROJECT_DIR, directory),
        exist_ok=True
    )

print("Project directories created successfully!")

Project directories created successfully!


In [3]:
!ls -R /content/drive/MyDrive/Load-Aware-SAGE

/content/drive/MyDrive/Load-Aware-SAGE:
artifacts  experiments

/content/drive/MyDrive/Load-Aware-SAGE/artifacts:
dataset  indexes  metadata  models

/content/drive/MyDrive/Load-Aware-SAGE/artifacts/dataset:
nq_clean_sample.jsonl  question_records.jsonl
nq_test_sample.jsonl   unique_documents.jsonl

/content/drive/MyDrive/Load-Aware-SAGE/artifacts/indexes:
bm25  faiss

/content/drive/MyDrive/Load-Aware-SAGE/artifacts/indexes/bm25:

/content/drive/MyDrive/Load-Aware-SAGE/artifacts/indexes/faiss:

/content/drive/MyDrive/Load-Aware-SAGE/artifacts/metadata:

/content/drive/MyDrive/Load-Aware-SAGE/artifacts/models:
random_forest

/content/drive/MyDrive/Load-Aware-SAGE/artifacts/models/random_forest:

/content/drive/MyDrive/Load-Aware-SAGE/experiments:
logs  oracle_labels

/content/drive/MyDrive/Load-Aware-SAGE/experiments/logs:

/content/drive/MyDrive/Load-Aware-SAGE/experiments/oracle_labels:


In [4]:
# ==========================================
# LOAD-AWARE SAGE: ORIGINAL BASELINE
# Configuration based on the research paper
# ==========================================

import os

PROJECT_DIR = "/content/drive/MyDrive/Load-Aware-SAGE"
ARTIFACT_DIR = os.path.join(PROJECT_DIR, "artifacts")

# Dataset
DATASET_NAME = "Natural Questions"
CORPUS_NAME = "English Wikipedia"

# Retrieval configuration
SPARSE_RETRIEVER = "BM25"
DENSE_RETRIEVER = "BGE-M3"
FUSION_METHOD = "RRF"

# Candidate retrieval budgets
K_VALUES = [2, 3, 5, 7, 10, 15, 20, 25, 30]

# SLO configuration
TARGET_P95_LATENCY = 5.0  # seconds

# Random Forest configuration
RF_N_ESTIMATORS = 100
RF_MAX_DEPTH = 10
RF_RANDOM_STATE = 42

# Probe retrieval
PROBE_LATENCY_TARGET = 0.300  # approximately 300 ms

print("SAGE baseline configuration loaded.")
print("Candidate budgets:", K_VALUES)
print("Target P95 latency:", TARGET_P95_LATENCY, "seconds")

SAGE baseline configuration loaded.
Candidate budgets: [2, 3, 5, 7, 10, 15, 20, 25, 30]
Target P95 latency: 5.0 seconds


In [5]:
# Install dataset preparation libraries

!pip install -q datasets pandas tqdm pyarrow

In [6]:
import datasets
import pandas as pd
import pyarrow

print("Dataset tools installed successfully!")
print("Datasets version:", datasets.__version__)
print("Pandas version:", pd.__version__)

Dataset tools installed successfully!
Datasets version: 4.8.5
Pandas version: 2.2.3


In [7]:

from datasets import load_dataset
from itertools import islice
import json
import os

SAMPLE_SIZE = 100

# Load Natural Questions in streaming mode
nq_stream = load_dataset(
    "google-research-datasets/natural_questions",
    split="train",
    streaming=True
)

# Output location
output_file = os.path.join(
    ARTIFACT_DIR,
    "dataset",
    "nq_test_sample.jsonl"
)

count = 0

with open(output_file, "w", encoding="utf-8") as f:
    for record in islice(nq_stream, SAMPLE_SIZE):
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        count += 1

print("Dataset download completed!")
print("Total records downloaded:", count)
print("Saved at:", output_file)

README.md:   0%|          | 0.00/13.7k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Dataset download completed!
Total records downloaded: 100
Saved at: /content/drive/MyDrive/Load-Aware-SAGE/artifacts/dataset/nq_test_sample.jsonl


In [8]:
import json

records = []

with open(output_file, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print("Total records:", len(records))

print("\nFirst question:")
print(records[0]["question"]["text"])

print("\nWikipedia document:")
print(records[0]["document"]["title"])

print("\nDataset fields:")
print(records[0].keys())

Total records: 100

First question:
when is the last episode of season 8 of the walking dead

Wikipedia document:
The Walking Dead (season 8)

Dataset fields:
dict_keys(['id', 'document', 'question', 'long_answer_candidates', 'annotations'])


In [9]:
!pip install -q beautifulsoup4

In [10]:
from bs4 import BeautifulSoup
import re

def clean_wikipedia_html(html):
    soup = BeautifulSoup(html, "html.parser")

    # 1. Extract infobox information
    infobox_text = ""

    infobox = soup.select_one("table.infobox")

    if infobox:
        fields = []

        for row in infobox.select("tr"):
            label = row.find("th")
            value = row.find("td")

            if label and value:
                key = label.get_text(" ", strip=True)
                val = value.get_text(" ", strip=True)

                if key and val:
                    fields.append(f"{key}: {val}")

        infobox_text = "\n".join(fields)
        infobox.decompose()

    # 2. Remove unwanted HTML elements
    for tag in soup([
        "script", "style", "nav", "footer",
        "header", "aside", "form", "button",
        "svg", "noscript"
    ]):
        tag.decompose()

    # 3. Remove unwanted Wikipedia elements
    for element in soup.select(
        ".mw-jump-link, .mw-editsection, "
        ".mw-indicators, .toc, #toc, "
        ".navbox, .vertical-navbox, "
        ".catlinks, .hatnote"
    ):
        element.decompose()

    # 4. Extract article text
    document_text = soup.get_text(
        separator=" ",
        strip=True
    )

    # Normalize extra spaces and line breaks
    document_text = re.sub(r"\s+", " ", document_text).strip()

    # 5. Remove unwanted Wikipedia text
    document_text = re.sub(
        r"From Wikipedia, the free encyclopedia",
        "",
        document_text
    )

    document_text = re.sub(
        r"Jump to:\s*navigation\s*,?\s*search",
        "",
        document_text,
        flags=re.IGNORECASE
    )

    # 6. Combine infobox and article content
    if infobox_text:
        document_text = (
            "ARTICLE INFORMATION:\n"
            + infobox_text
            + "\n\nARTICLE CONTENT:\n"
            + document_text
        )

    document_text = re.sub(r"\n+", "\n", document_text)
    document_text = re.sub(r"[ \t]+", " ", document_text)

    return document_text.strip()

In [11]:
clean_records = []

for record in records:
    document_text = clean_wikipedia_html(
        record["document"]["html"]
    )

    clean_records.append({
        "question": record["question"]["text"],
        "document_title": record["document"]["title"],
        "document_text": document_text,
        "annotations": record["annotations"]
    })

print("Total cleaned records:", len(clean_records))
print("\nFirst 1000 characters:\n")
print(clean_records[0]["document_text"][:1000])

Total cleaned records: 100

First 1000 characters:

ARTICLE INFORMATION:
Starring: Andrew Lincoln Norman Reedus Lauren Cohan Chandler Riggs Danai Gurira Melissa McBride Lennie James Alanna Masterson Josh McDermitt Christian Serratos Seth Gilliam Ross Marquand Jeffrey Dean Morgan Austin Amelio Tom Payne Xander Berkeley Khary Payton Steven Ogg Katelyn Nacon Pollyanna McIntosh
Country of origin: United States
No. of episodes: 9
Original network: AMC
Original release: October 22, 2017 ( 2017-10-22 ) – present ( present )
ARTICLE CONTENT:
The Walking Dead (season 8) - Wikipedia The Walking Dead (season 8) The eighth season of The Walking Dead , an American post-apocalyptic horror television series on AMC , premiered on October 22, 2017, [1] and will consist of 16 episodes [2] split into two eight-episode parts, with the second part debuting on February 25, 2018. [3] Developed for television by Frank Darabont , the series is based on the eponymous series of comic books by Robert Kirkman , To

In [12]:
sample = clean_records[0]

print("QUESTION:")
print(sample["question"])

print("\nDOCUMENT TITLE:")
print(sample["document_title"])

print("\nEXTRACTED TEXT:")
print(sample["document_text"][:1500])

print("\nDOCUMENT LENGTH:")
print(len(sample["document_text"]), "characters")

QUESTION:
when is the last episode of season 8 of the walking dead

DOCUMENT TITLE:
The Walking Dead (season 8)

EXTRACTED TEXT:
ARTICLE INFORMATION:
Starring: Andrew Lincoln Norman Reedus Lauren Cohan Chandler Riggs Danai Gurira Melissa McBride Lennie James Alanna Masterson Josh McDermitt Christian Serratos Seth Gilliam Ross Marquand Jeffrey Dean Morgan Austin Amelio Tom Payne Xander Berkeley Khary Payton Steven Ogg Katelyn Nacon Pollyanna McIntosh
Country of origin: United States
No. of episodes: 9
Original network: AMC
Original release: October 22, 2017 ( 2017-10-22 ) – present ( present )
ARTICLE CONTENT:
The Walking Dead (season 8) - Wikipedia The Walking Dead (season 8) The eighth season of The Walking Dead , an American post-apocalyptic horror television series on AMC , premiered on October 22, 2017, [1] and will consist of 16 episodes [2] split into two eight-episode parts, with the second part debuting on February 25, 2018. [3] Developed for television by Frank Darabont , the 

In [13]:
import json

sample = clean_records[0]

print("QUESTION:")
print(sample["question"])

print("\nANNOTATIONS:")
print(json.dumps(
    sample["annotations"],
    indent=2
))

QUESTION:
when is the last episode of season 8 of the walking dead

ANNOTATIONS:
{
  "id": [
    "6782080525527814293"
  ],
  "long_answer": [
    {
      "candidate_index": 92,
      "end_byte": 96948,
      "end_token": 3538,
      "start_byte": 82798,
      "start_token": 2114
    }
  ],
  "short_answers": [
    {
      "end_byte": [
        96731
      ],
      "end_token": [
        3525
      ],
      "start_byte": [
        96715
      ],
      "start_token": [
        3521
      ],
      "text": [
        "March 18, 2018"
      ]
    }
  ],
  "yes_no_answer": [
    -1
  ]
}


In [14]:
for i in range(5):
    sample = clean_records[i]

    print(f"\nQUESTION {i+1}:")
    print(sample["question"])

    print("SHORT ANSWER:")
    print(sample["annotations"]["short_answers"][0]["text"])

    print("-" * 60)


QUESTION 1:
when is the last episode of season 8 of the walking dead
SHORT ANSWER:
['March 18, 2018']
------------------------------------------------------------

QUESTION 2:
in greek mythology who was the goddess of spring growth
SHORT ANSWER:
['Persephone (/pərˈsɛfəni/; Greek: Περσεφόνη), also called Kore (/ˈkɔːriː/; "the maiden")']
------------------------------------------------------------

QUESTION 3:
benefits of colonial life for single celled organisms
SHORT ANSWER:
[]
------------------------------------------------------------

QUESTION 4:
how many season of the man in the high castle
SHORT ANSWER:
[]
------------------------------------------------------------

QUESTION 5:
who was the first ministry head of state in nigeria
SHORT ANSWER:
[]
------------------------------------------------------------


In [15]:
total = len(clean_records)

short_count = 0
long_count = 0
yes_no_count = 0

for record in clean_records:
    annotation = record["annotations"]

    if annotation["short_answers"][0]["text"]:
        short_count += 1

    if annotation["long_answer"][0]["start_token"] >= 0:
        long_count += 1

    if annotation["yes_no_answer"][0] != -1:
        yes_no_count += 1

print("Total records:", total)
print("With short answers:", short_count)
print("With long answers:", long_count)
print("With yes/no answers:", yes_no_count)
print("Without short answers:", total - short_count)

Total records: 100
With short answers: 41
With long answers: 56
With yes/no answers: 1
Without short answers: 59


In [16]:
clean_file = os.path.join(
    ARTIFACT_DIR,
    "dataset",
    "nq_clean_sample.jsonl"
)

with open(clean_file, "w", encoding="utf-8") as f:
    for record in clean_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

In [17]:
import os
import json

clean_file = os.path.join(
    ARTIFACT_DIR,
    "dataset",
    "nq_clean_sample.jsonl"
)

if os.path.exists(clean_file):
    with open(clean_file, "r", encoding="utf-8") as f:
        clean_records = [json.loads(line) for line in f]

    print("Dataset found!")
    print("Total records:", len(clean_records))
    print("First question:", clean_records[0]["question"])
    print("Document title:", clean_records[0]["document_title"])
    print("Document characters:", len(clean_records[0]["document_text"]))
    print("Annotations available:", "annotations" in clean_records[0])
else:
    print("Cleaned dataset not found. Run the save cell first.")

Dataset found!
Total records: 100
First question: when is the last episode of season 8 of the walking dead
Document title: The Walking Dead (season 8)
Document characters: 22820
Annotations available: True


In [18]:
import hashlib
import json
import os

unique_documents = {}
question_records = []

for record in clean_records:

    document_text = record["document_text"].strip()

    # Generate a unique hash for the document
    doc_hash = hashlib.sha256(
        document_text.encode("utf-8")
    ).hexdigest()

    # Check whether this document already exists
    if doc_hash not in unique_documents:

        doc_id = f"doc_{len(unique_documents) + 1}"

        unique_documents[doc_hash] = {
            "doc_id": doc_id,
            "document_title": record["document_title"],
            "document_text": document_text
        }

    else:
        doc_id = unique_documents[doc_hash]["doc_id"]

    # Preserve every question and its annotations
    question_records.append({
        "question": record["question"],
        "doc_id": doc_id,
        "annotations": record["annotations"]
    })

# Convert dictionary into a list
unique_documents = list(unique_documents.values())

# Save unique documents
documents_file = os.path.join(
    ARTIFACT_DIR,
    "dataset",
    "unique_documents.jsonl"
)

with open(documents_file, "w", encoding="utf-8") as f:
    for doc in unique_documents:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

# Save question records
questions_file = os.path.join(
    ARTIFACT_DIR,
    "dataset",
    "question_records.jsonl"
)

with open(questions_file, "w", encoding="utf-8") as f:
    for item in question_records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Original records:", len(clean_records))
print("Unique documents:", len(unique_documents))
print("Duplicate documents:", len(clean_records) - len(unique_documents))
print("Question records preserved:", len(question_records))
print("Files saved successfully!")

Original records: 100
Unique documents: 100
Duplicate documents: 0
Question records preserved: 100
Files saved successfully!


In [19]:
!pip install -q langchain-text-splitters

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "? ",
        "! ",
        "; ",
        ", ",
        " ",
        ""
    ],
    length_function=len,
    is_separator_regex=False
)

In [21]:
all_chunks = []

for doc in unique_documents:

    chunks = text_splitter.split_text(
        doc["document_text"]
    )

    for i, chunk in enumerate(chunks):

        all_chunks.append({
            "chunk_id": f'{doc["doc_id"]}_chunk_{i+1}',
            "doc_id": doc["doc_id"],
            "document_title": doc["document_title"],
            "chunk_text": chunk
        })

print("Total documents:", len(unique_documents))
print("Total chunks:", len(all_chunks))

Total documents: 100
Total chunks: 8826


In [22]:
# Display the first 5 chunks

for chunk in all_chunks[:5]:

    print("Chunk ID:", chunk["chunk_id"])
    print("Document ID:", chunk["doc_id"])
    print("Chunk characters:", len(chunk["chunk_text"]))
    print("Chunk text:", chunk["chunk_text"][:300])
    print("-" * 80)

Chunk ID: doc_1_chunk_1
Document ID: doc_1
Chunk characters: 487
Chunk text: ARTICLE INFORMATION:
Starring: Andrew Lincoln Norman Reedus Lauren Cohan Chandler Riggs Danai Gurira Melissa McBride Lennie James Alanna Masterson Josh McDermitt Christian Serratos Seth Gilliam Ross Marquand Jeffrey Dean Morgan Austin Amelio Tom Payne Xander Berkeley Khary Payton Steven Ogg Katelyn 
--------------------------------------------------------------------------------
Chunk ID: doc_1_chunk_2
Document ID: doc_1
Chunk characters: 489
Chunk text: The Walking Dead (season 8) - Wikipedia The Walking Dead (season 8) The eighth season of The Walking Dead , an American post-apocalyptic horror television series on AMC , premiered on October 22, 2017, [1] and will consist of 16 episodes [2] split into two eight-episode parts, with the second part d
--------------------------------------------------------------------------------
Chunk ID: doc_1_chunk_3
Document ID: doc_1
Chunk characters: 453
Chunk text: . The

In [23]:
chunk_sizes = [
    len(chunk["chunk_text"])
    for chunk in all_chunks
]

print("Total chunks:", len(chunk_sizes))
print("Smallest chunk:", min(chunk_sizes))
print("Largest chunk:", max(chunk_sizes))
print("Average chunk size:", sum(chunk_sizes) / len(chunk_sizes))

print("Chunks above 500 characters:",
      sum(size > 500 for size in chunk_sizes))

Total chunks: 8826
Smallest chunk: 3
Largest chunk: 500
Average chunk size: 409.5748923634716
Chunks above 500 characters: 0


In [24]:
!pip install -q FlagEmbedding rank-bm25 faiss-cpu

In [25]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel(
    'BAAI/bge-m3',
    use_fp16=True
)

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [26]:
import numpy as np

chunk_texts = [
    chunk["chunk_text"]
    for chunk in all_chunks
]

embeddings = model.encode(
    chunk_texts,
    batch_size=8,
    max_length=512
)["dense_vecs"]

embeddings = np.asarray(embeddings, dtype=np.float32)

print("Embedding shape:", embeddings.shape)



pre tokenize:   0%|          | 0/1104 [00:00<?, ?it/s]

pre tokenize:   2%|▏         | 20/1104 [00:00<00:05, 193.35it/s]

pre tokenize:   4%|▎         | 41/1104 [00:00<00:05, 201.59it/s]

pre tokenize:   6%|▌         | 64/1104 [00:00<00:04, 213.62it/s]

pre tokenize:   8%|▊         | 87/1104 [00:00<00:04, 219.97it/s]

pre tokenize:  10%|▉         | 110/1104 [00:00<00:04, 210.66it/s]

pre tokenize:  12%|█▏        | 132/1104 [00:00<00:05, 192.20it/s]

pre tokenize:  14%|█▍        | 152/1104 [00:00<00:05, 184.37it/s]

pre tokenize:  16%|█▌        | 172/1104 [00:00<00:04, 188.31it/s]

pre tokenize:  17%|█▋        | 192/1104 [00:00<00:04, 186.82it/s]

pre tokenize:  19%|█▉        | 212/1104 [00:01<00:04, 190.06it/s]

pre tokenize:  21%|██        | 232/1104 [00:01<00:04, 188.88it/s]

pre tokenize:  23%|██▎       | 253/1104 [00:01<00:04, 192.46it/s]

pre tokenize:  25%|██▌       | 276/1104 [00:01<00:04, 201.65it/s]

pre tokenize:  27%|██▋       | 299/1104 [00:01<00:03, 207.48it/s]

pre toke

Embedding shape: (8826, 1024)


In [27]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

faiss.normalize_L2(embeddings)

index.add(embeddings)

print("Total vectors stored:", index.ntotal)

Total vectors stored: 8826


In [28]:
query = question_records[1]["question"]

query_embedding = model.encode(
    [query],
    max_length=512
)["dense_vecs"]

query_embedding = np.asarray(
    query_embedding,
    dtype=np.float32
)

faiss.normalize_L2(query_embedding)

scores, indices = index.search(query_embedding, k=5)

for rank, (idx, score) in enumerate(
    zip(indices[0], scores[0]), start=1
):
    print(f"\nRank: {rank}")
    print(f"Similarity Score: {score:.4f}")
    print(f"Chunk: {all_chunks[idx]['chunk_text']}")


Rank: 1
Similarity Score: 0.5832
Chunk: . Plutarch identifies her with spring and Cicero calls her the seed of the fruits of the fields. In the Eleusinian mysteries, her return is the symbol of immortality and hence she was frequently represented on sarcophagi

Rank: 2
Similarity Score: 0.5586
Chunk: . He pursues the mare-Demeter and from the union she bears the horse Arion and a daughter who originally had the form or the shape of a mare. The two goddesses were not clearly separated and they were closely connected with the springs and the animals. They were related with the god of rivers and springs; Poseidon and especially with Artemis , the Mistress of the Animals who was the first nymph . [1] According to the Greek tradition a hunt-goddess preceded the harvest goddess

Rank: 3
Similarity Score: 0.5318
Chunk: . Homer describes her as the formidable, venerable majestic princess of the underworld, who carries into effect the curses of men upon the souls of the dead. Persephone was ma

In [29]:
from rank_bm25 import BM25Okapi
import re

chunk_texts = [
    chunk["chunk_text"]
    for chunk in all_chunks
]

tokenized_chunks = [
    re.findall(r"\b\w+\b", text.lower())
    for text in chunk_texts
]

In [30]:
bm25 = BM25Okapi(tokenized_chunks)

print("BM25 index created successfully!")
print("Total documents indexed:", len(tokenized_chunks))

BM25 index created successfully!
Total documents indexed: 8826


In [31]:
query = question_records[1]["question"]

tokenized_query = re.findall(
    r"\b\w+\b", query.lower()
)

bm25_scores = bm25.get_scores(tokenized_query)

top_k = 5
top_indices = bm25_scores.argsort()[-top_k:][::-1]

for rank, idx in enumerate(top_indices, start=1):
    print(f"\nRank: {rank}")
    print(f"BM25 Score: {bm25_scores[idx]:.4f}")
    print(f"Chunk: {all_chunks[idx]['chunk_text']}")


Rank: 1
BM25 Score: 28.4174
Chunk: . [54] As goddess of death she was also called a daughter of Zeus and Styx , [75] the river that formed the boundary between Earth and the underworld. Homer describes her as the formidable, venerable majestic queen of the shades, who carries into effect the curses of men upon the souls of the dead, along with her husband Hades. [76] In the reformulation of Greek mythology expressed in the Orphic Hymns , Dionysus and Melinoe are separately called children of Zeus and Persephone

Rank: 2
BM25 Score: 27.8598
Chunk: . Edmonds III, "Orphic Mythology," in A Companion to Greek Mythology (Blackwell, 2011), p. 100. ^ Jump up to: a b c d e f g h i j k l m n Rhode (1961), Psyche I, pp. 206–210 ^ Jump up to: a b Nilsson (1967) Vol I, pp. 478–480 Jump up ^ Orphic Hymn 29 to Persephone Jump up ^ "PERSEPHONE - Greek Goddess of Spring, Queen of the Underworld (Roman Proserpina)" . ^ Jump up to: a b c d e f g h i j k Nilsson (1967) Vol I, pp. 463–466 ^ Jump up to: a 

In [32]:
dense_ranked_ids = indices[0].tolist()

sparse_ranked_ids = top_indices.tolist()

In [33]:
def reciprocal_rank_fusion(
    dense_ranked_ids,
    sparse_ranked_ids,
    k=60
):
    rrf_scores = {}

    # Add FAISS ranking contributions
    for rank, chunk_id in enumerate(dense_ranked_ids, start=1):
        rrf_scores[chunk_id] = (
            rrf_scores.get(chunk_id, 0) + 1 / (k + rank)
        )

    # Add BM25 ranking contributions
    for rank, chunk_id in enumerate(sparse_ranked_ids, start=1):
        rrf_scores[chunk_id] = (
            rrf_scores.get(chunk_id, 0) + 1 / (k + rank)
        )

    # Sort by highest RRF score
    ranked_results = sorted(
        rrf_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked_results

In [34]:
# Combine FAISS and BM25 rankings
hybrid_results = reciprocal_rank_fusion(
    dense_ranked_ids,
    sparse_ranked_ids,
    k=60
)

# Display hybrid rankings
print("Hybrid Retrieval Results:\n")

for rank, (chunk_id, score) in enumerate(hybrid_results, start=1):
    print(
        f"Rank: {rank} | "
        f"Chunk ID: {chunk_id} | "
        f"RRF Score: {score:.6f}"
    )

Hybrid Retrieval Results:

Rank: 1 | Chunk ID: 77 | RRF Score: 0.016393
Rank: 2 | Chunk ID: 125 | RRF Score: 0.016393
Rank: 3 | Chunk ID: 122 | RRF Score: 0.016129
Rank: 4 | Chunk ID: 173 | RRF Score: 0.016129
Rank: 5 | Chunk ID: 63 | RRF Score: 0.015873
Rank: 6 | Chunk ID: 83 | RRF Score: 0.015873
Rank: 7 | Chunk ID: 64 | RRF Score: 0.015625
Rank: 8 | Chunk ID: 98 | RRF Score: 0.015625
Rank: 9 | Chunk ID: 85 | RRF Score: 0.015385
Rank: 10 | Chunk ID: 123 | RRF Score: 0.015385


In [35]:
for rank, (chunk_id, score) in enumerate(hybrid_results, start=1):

    print(f"\n{'='*60}")
    print(f"Rank: {rank}")
    print(f"Chunk ID: {chunk_id}")
    print(f"RRF Score: {score:.6f}")

    print("Chunk Text:")
    print(all_chunks[chunk_id]["chunk_text"])


Rank: 1
Chunk ID: 77
RRF Score: 0.016393
Chunk Text:
. Plutarch identifies her with spring and Cicero calls her the seed of the fruits of the fields. In the Eleusinian mysteries, her return is the symbol of immortality and hence she was frequently represented on sarcophagi

Rank: 2
Chunk ID: 125
RRF Score: 0.016393
Chunk Text:
. [54] As goddess of death she was also called a daughter of Zeus and Styx , [75] the river that formed the boundary between Earth and the underworld. Homer describes her as the formidable, venerable majestic queen of the shades, who carries into effect the curses of men upon the souls of the dead, along with her husband Hades. [76] In the reformulation of Greek mythology expressed in the Orphic Hymns , Dionysus and Melinoe are separately called children of Zeus and Persephone

Rank: 3
Chunk ID: 122
RRF Score: 0.016129
Chunk Text:
. He pursues the mare-Demeter and from the union she bears the horse Arion and a daughter who originally had the form or the shape of

In [36]:
print("Question:")
print(question_records[1]["question"])

print("\nAnnotations:")
print(question_records[1]["annotations"])

Question:
in greek mythology who was the goddess of spring growth

Annotations:
{'id': ['7719528322202775345'], 'long_answer': [{'candidate_index': 58, 'end_byte': 84070, 'end_token': 965, 'start_byte': 81195, 'start_token': 809}], 'short_answers': [{'end_byte': [82556], 'end_token': [838], 'start_byte': [81281], 'start_token': [814], 'text': ['Persephone (/pərˈsɛfəni/; Greek: Περσεφόνη), also called Kore (/ˈkɔːriː/; "the maiden")']}], 'yes_no_answer': [-1]}


In [37]:
ground_truth = []

for record in question_records:
    answers = []

    for item in record["annotations"]["short_answers"]:
        answers.extend(item["text"])

    ground_truth.append(answers)

print("Question:", question_records[1]["question"])
print("Ground-truth:", ground_truth[1])

Question: in greek mythology who was the goddess of spring growth
Ground-truth: ['Persephone (/pərˈsɛfəni/; Greek: Περσεφόνη), also called Kore (/ˈkɔːriː/; "the maiden")']


In [38]:
answer = "Persephone"

def answer_hit_at_k(ranked_ids, k):
    top_chunks = ranked_ids[:k]

    for chunk_id in top_chunks:
        chunk_text = all_chunks[chunk_id]["chunk_text"]

        if answer.lower() in chunk_text.lower():
            return 1

    return 0


dense_ids = dense_ranked_ids
sparse_ids = sparse_ranked_ids
hybrid_ids = [chunk_id for chunk_id, score in hybrid_results]

for k in [1, 3, 5]:
    print(f"\nResults at K = {k}")

    print("Dense Hit@K:",
          answer_hit_at_k(dense_ids, k))

    print("BM25 Hit@K:",
          answer_hit_at_k(sparse_ids, k))

    print("Hybrid Hit@K:",
          answer_hit_at_k(hybrid_ids, k))


Results at K = 1
Dense Hit@K: 0
BM25 Hit@K: 1
Hybrid Hit@K: 0

Results at K = 3
Dense Hit@K: 1
BM25 Hit@K: 1
Hybrid Hit@K: 1

Results at K = 5
Dense Hit@K: 1
BM25 Hit@K: 1
Hybrid Hit@K: 1


In [39]:
def reciprocal_rank(ranked_ids):
    for rank, chunk_id in enumerate(ranked_ids, start=1):
        chunk_text = all_chunks[chunk_id]["chunk_text"]

        if answer.lower() in chunk_text.lower():
            return 1 / rank

    return 0


print("Dense MRR:", reciprocal_rank(dense_ids))
print("BM25 MRR:", reciprocal_rank(sparse_ids))
print("Hybrid MRR:", reciprocal_rank(hybrid_ids))

Dense MRR: 0.3333333333333333
BM25 MRR: 1.0
Hybrid MRR: 0.5


In [40]:
print("Total questions:", len(question_records))

print("\nFirst question:")
print(question_records[0]["question"])

print("\nFirst document ID:")
print(question_records[0]["doc_id"])

print("\nAvailable keys:")
print(question_records[0].keys())

Total questions: 100

First question:
when is the last episode of season 8 of the walking dead

First document ID:
doc_1

Available keys:
dict_keys(['question', 'doc_id', 'annotations'])


In [41]:
def dense_retrieval(query, top_k=10):

    # Convert the query into an embedding
    query_embedding = model.encode(
        [query],
        max_length=512
    )["dense_vecs"]

    # Convert embedding into float32
    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )

    # Normalize the query vector
    faiss.normalize_L2(query_embedding)

    # Search for the top K similar chunks
    scores, indices = index.search(
        query_embedding,
        top_k
    )

    return indices[0].tolist()

In [42]:
test_query = question_records[0]["question"]

dense_results = dense_retrieval(test_query, top_k=5)

print("Question:", test_query)
print("\nRetrieved Chunk IDs:", dense_results)

for rank, chunk_id in enumerate(dense_results, start=1):
    print(f"\nRank {rank} | Chunk ID: {chunk_id}")
    print(all_chunks[chunk_id]["chunk_text"][:300])

Question: when is the last episode of season 8 of the walking dead

Retrieved Chunk IDs: [1, 19, 37, 38, 55]

Rank 1 | Chunk ID: 1
The Walking Dead (season 8) - Wikipedia The Walking Dead (season 8) The eighth season of The Walking Dead , an American post-apocalyptic horror television series on AMC , premiered on October 22, 2017, [1] and will consist of 16 episodes [2] split into two eight-episode parts, with the second part d

Rank 2 | Chunk ID: 19
. Miscellaneous Avi Nash as Siddiq, a mysterious vagabond survivor who befriends Carl. Production The Walking Dead was renewed by AMC for a 16-episode eighth season on October 16, 2016, [2] and that same month it was reported that Maria Bello would have a role on the show. [6] Production began on Ap

Rank 3 | Chunk ID: 37
. The second trailer was released on February 1, 2018, for the second part of the season. [22] Reception Critical response The eighth season of The Walking Dead has received mostly positive reviews from critics. On Rotten

In [43]:
def bm25_retrieval(query, top_k=10):

    # Tokenize the question
    tokenized_query = re.findall(
        r"\b\w+\b",
        query.lower()
    )

    # Calculate BM25 scores
    scores = bm25.get_scores(tokenized_query)

    # Get indices of top K scores
    top_indices = scores.argsort()[-top_k:][::-1]

    return top_indices.tolist()

In [44]:
test_query = question_records[0]["question"]

bm25_results = bm25_retrieval(test_query, top_k=5)

print("Question:", test_query)
print("\nRetrieved Chunk IDs:", bm25_results)

for rank, chunk_id in enumerate(bm25_results, start=1):
    print(f"\nRank {rank} | Chunk ID: {chunk_id}")
    print(all_chunks[chunk_id]["chunk_text"][:300])

Question: when is the last episode of season 8 of the walking dead

Retrieved Chunk IDs: [1, 38, 20, 50, 43]

Rank 1 | Chunk ID: 1
The Walking Dead (season 8) - Wikipedia The Walking Dead (season 8) The eighth season of The Walking Dead , an American post-apocalyptic horror television series on AMC , premiered on October 22, 2017, [1] and will consist of 16 episodes [2] split into two eight-episode parts, with the second part d

Rank 2 | Chunk ID: 38
. The site's critical consensus reads: " The Walking Dead ' s eighth season energizes its characters with some much-needed angst and action, though it's still occasionally choppy and lacking forward-moving plot progression." [23] The Walking Dead (season 8): Critical reception by episode Season 8 (2

Rank 3 | Chunk ID: 20
. [8] Production resumed on July 17. [9] The season premiere, which also serves as the series' milestone 100th episode , was directed by executive producer Greg Nicotero . [10] In November 2017, it was announced that Lenn

In [45]:
def hybrid_retrieval(query, top_k=10, rrf_k=60):

    # Get Dense results
    dense_ids = dense_retrieval(query, top_k)

    # Get BM25 results
    sparse_ids = bm25_retrieval(query, top_k)

    # Combine using RRF
    hybrid_results = reciprocal_rank_fusion(
        dense_ids,
        sparse_ids,
        k=rrf_k
    )

    # Return top K chunk IDs
    return [
        chunk_id
        for chunk_id, score in hybrid_results[:top_k]
    ]

In [46]:
test_query = question_records[0]["question"]

hybrid_ids = hybrid_retrieval(test_query, top_k=5)

print("Question:", test_query)
print("\nHybrid Retrieved Chunk IDs:", hybrid_ids)

for rank, chunk_id in enumerate(hybrid_ids, start=1):
    print(f"\nRank {rank} | Chunk ID: {chunk_id}")
    print(all_chunks[chunk_id]["chunk_text"][:300])

Question: when is the last episode of season 8 of the walking dead

Hybrid Retrieved Chunk IDs: [1, 38, 19, 37, 20]

Rank 1 | Chunk ID: 1
The Walking Dead (season 8) - Wikipedia The Walking Dead (season 8) The eighth season of The Walking Dead , an American post-apocalyptic horror television series on AMC , premiered on October 22, 2017, [1] and will consist of 16 episodes [2] split into two eight-episode parts, with the second part d

Rank 2 | Chunk ID: 38
. The site's critical consensus reads: " The Walking Dead ' s eighth season energizes its characters with some much-needed angst and action, though it's still occasionally choppy and lacking forward-moving plot progression." [23] The Walking Dead (season 8): Critical reception by episode Season 8 (2

Rank 3 | Chunk ID: 19
. Miscellaneous Avi Nash as Siddiq, a mysterious vagabond survivor who befriends Carl. Production The Walking Dead was renewed by AMC for a 16-episode eighth season on October 16, 2016, [2] and that same month it w

In [47]:
def get_ground_truth(record):

    answers = []

    for item in record["annotations"]["short_answers"]:
        answers.extend(item["text"])

    return answers

In [48]:
for i in range(3):

    answers = get_ground_truth(question_records[i])

    print("Question:", question_records[i]["question"])
    print("Ground truth:", answers)
    print("-" * 60)

Question: when is the last episode of season 8 of the walking dead
Ground truth: ['March 18, 2018']
------------------------------------------------------------
Question: in greek mythology who was the goddess of spring growth
Ground truth: ['Persephone (/pərˈsɛfəni/; Greek: Περσεφόνη), also called Kore (/ˈkɔːriː/; "the maiden")']
------------------------------------------------------------
Question: benefits of colonial life for single celled organisms
Ground truth: []
------------------------------------------------------------


In [49]:
evaluation_records = []

for record in question_records:

    answers = get_ground_truth(record)

    if answers:
        evaluation_records.append({
            "question": record["question"],
            "doc_id": record["doc_id"],
            "answers": answers
        })

print("Total questions:", len(question_records))
print("Questions with ground truth:", len(evaluation_records))
print("Questions skipped:", len(question_records) - len(evaluation_records))

Total questions: 100
Questions with ground truth: 41
Questions skipped: 59


In [50]:
# Test hybrid retrieval on the first 5 evaluation questions

for i, record in enumerate(evaluation_records[:5]):
    question = record["question"]

    results = hybrid_retrieval(question, top_k=5)

    print(f"\nQuestion {i+1}: {question}")
    print("Retrieved chunk IDs:", results)
    print("Ground truth:", record["answers"])


Question 1: when is the last episode of season 8 of the walking dead
Retrieved chunk IDs: [1, 38, 19, 37, 20]
Ground truth: ['March 18, 2018']

Question 2: in greek mythology who was the goddess of spring growth
Retrieved chunk IDs: [77, 125, 122, 173, 63]
Ground truth: ['Persephone (/pərˈsɛfəni/; Greek: Περσεφόνη), also called Kore (/ˈkɔːriː/; "the maiden")']

Question 3: what is the name of the most important jewish text
Retrieved chunk IDs: [835, 879, 869, 1571, 875]
Ground truth: ['the Shulchan Aruch']

Question 4: what is the name of spain's most famous soccer team
Retrieved chunk IDs: [1224, 1281, 1299, 1235, 1254]
Ground truth: ['Real Madrid']

Question 5: when was the first robot used in surgery
Retrieved chunk IDs: [1370, 1372, 1368, 1351, 1358]
Ground truth: ['1983']


In [51]:
def answer_hit_at_k(ranked_ids, answers, k):
    top_chunks = ranked_ids[:k]

    for chunk_id in top_chunks:
        chunk_text = all_chunks[chunk_id]["chunk_text"].lower()

        for answer in answers:
            if answer.lower() in chunk_text:
                return 1

    return 0

In [52]:
hit_results = {
    1: [],
    3: [],
    5: []
}

for record in evaluation_records:
    question = record["question"]
    answers = record["answers"]

    retrieved_ids = hybrid_retrieval(question, top_k=10)

    for k in [1, 3, 5]:
        hit = answer_hit_at_k(retrieved_ids, answers, k)
        hit_results[k].append(hit)

for k in [1, 3, 5]:
    score = sum(hit_results[k]) / len(hit_results[k])

    print(f"Hit@{k}: {score:.4f} ({score * 100:.2f}%)")

Hit@1: 0.4878 (48.78%)
Hit@3: 0.7073 (70.73%)
Hit@5: 0.7805 (78.05%)
